<a href="https://colab.research.google.com/github/pulindu-seniya-silva/ASAS_FDM/blob/Logistic_Regression_Indhi/Logistic_Regression_Indhi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Pre-processing

Importing relevent libraries

Importing dataset

Keeping save copies of the dataset

In [1]:
import os, re, json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy import sparse
import joblib

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    balanced_accuracy_score, recall_score
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV

CSV_PATH = "Road Accident Data.csv"   # drag and drop dataset directly to the files in colab and keep this as it is
OUTDIR   = "artifacts"
TEST_SIZE = 0.20
RANDOM_STATE = 42

os.makedirs(OUTDIR, exist_ok=True)

# load and keep a safe raw copy
df_in  = pd.read_csv(CSV_PATH)
df_raw = df_in.copy(deep=True)   # audit/rollback
df     = df_in.copy(deep=True)   # working copy

print("Loaded:", df.shape)
print("Dataset:")
df.head(5)

def evaluate(y_true, y_pred, title=""):
    print(f"\n=== {title} ===")
    print("Macro F1:       ", round(f1_score(y_true, y_pred, average="macro"), 4))
    print("Recall (macro): ", round(recall_score(y_true, y_pred, average="macro"), 4))
    print("Balanced Acc:   ", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("\nClassification report:\n", classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))
    cm_norm = (cm.T / cm.sum(axis=1, where=cm.sum(axis=1)!=0)).T  # safe normalize
    print("Confusion matrix (row-normalized):\n", np.round(cm_norm, 3))



Loaded: (14045, 21)
Dataset:


drop purely irrelevant columns early

In [2]:
early_drop = [c for c in [
    "Accident_Index",
    "Local_Authority_(District)",
    "Police_Force",
    "Latitude", "Longitude",
    "Vehicle_Type",
    "Carriageway_Hazards"
] if c in df.columns]

df = df.drop(columns=early_drop, errors="ignore")
print("After early drop:", df.shape)
print("Dropped:", early_drop)


After early drop: (14045, 14)
Dropped: ['Accident_Index', 'Local_Authority_(District)', 'Police_Force', 'Latitude', 'Longitude', 'Vehicle_Type', 'Carriageway_Hazards']


Implimenting string hygene utils

In [3]:
def strip_collapse(s):
    if pd.isna(s): return s
    return re.sub(r"\s+", " ", str(s).strip())

def map_values(series, mapping):
    return series.replace(mapping)

def bucket_rare(series, min_count=100, other_label="Other"):
    vc = series.value_counts(dropna=False)
    rare = vc[vc < min_count].index
    return series.where(~series.isin(rare), other_label)

# trim whitespace on all object columns to avoid duplicate labels
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(strip_collapse)


**Basic cleaning**

- Fix tatget label typos and remove missing values

In [4]:
if "Accident_Severity" in df.columns:
    df["Accident_Severity"] = df["Accident_Severity"].astype("object")
    # from your counts: "Fetal" is a typo
    df["Accident_Severity"] = map_values(df["Accident_Severity"],
                                         {"Fetal": "Fatal", "fetal": "Fatal"})
    before = df.shape[0]
    df = df[~df["Accident_Severity"].isna()].copy()
    print("Dropped rows with missing Accident_Severity:", before - df.shape[0])


Dropped rows with missing Accident_Severity: 0


- parse "accident_date" and recompute "Day_of_Week"

In [5]:
if "Accident Date" in df.columns:
    df["Accident Date"] = pd.to_datetime(df["Accident Date"], errors="coerce", dayfirst=True)
    print("Unparseable dates:", df["Accident Date"].isna().sum())

    # recompute Day_of_Week from parsed date (consistent + fixes 1 NaN)
    df["Day_of_Week"] = df["Accident Date"].dt.day_name()


Unparseable dates: 8388


- normalize "Time" to HH:MM and flag invalid

In [6]:
if "Time" in df.columns:
    def normalize_time(x):
        if pd.isna(x): return np.nan
        s = str(x).strip()
        m = re.match(r"^(\d{1,2}):(\d{2})$", s)
        if not m: return np.nan
        hh, mm = int(m.group(1)), int(m.group(2))
        if 0 <= hh <= 23 and 0 <= mm <= 59:
            return f"{hh:02d}:{mm:02d}"
        return np.nan

    df["Time"] = df["Time"].apply(normalize_time)
    print("Invalid/NaN times:", df["Time"].isna().sum())


Invalid/NaN times: 1


- Harmonize "unknown" values and shorten long labels

In [7]:
# unify explicit "Unknown" labels
if "Junction_Control" in df.columns:
    df["Junction_Control"] = map_values(df["Junction_Control"],
                                        {"Data missing or out of range": "Unknown"})

if "Junction_Detail" in df.columns:
    df["Junction_Detail"] = map_values(df["Junction_Detail"],
                                       {"Not at junction or within 20 metres": "Not at junction"})

# fill NaN -> "Unknown" for key categoricals
for col in ["Light_Conditions","Weather_Conditions","Road_Surface_Conditions",
            "Road_Type","Junction_Control","Junction_Detail",
            "Urban_or_Rural_Area"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# Compact some verbose labels (optional but cleaner for OHE and plots)
if "Light_Conditions" in df.columns:
    df["Light_Conditions"] = map_values(df["Light_Conditions"], {
        "Darkness - lights lit": "Dark_lit",
        "Darkness - lights unlit": "Dark_unlit",
        "Darkness - no lighting": "Dark_none",
        "Darkness - lighting unknown": "Dark_unknown",
    })

if "Weather_Conditions" in df.columns:
    df["Weather_Conditions"] = map_values(df["Weather_Conditions"], {
        "Fine + high winds": "Fine_high_winds",
        "Raining + high winds": "Rain_high_winds",
        "Raining no high winds": "Rain_no_high",
        "Snowing + high winds": "Snow_high_winds",
        "Snowing no high winds": "Snow_no_high",
        "Fog or mist": "Fog_mist",
        "Fine no high winds": "Fine_no_high",
    })

if "Road_Surface_Conditions" in df.columns:
    df["Road_Surface_Conditions"] = map_values(df["Road_Surface_Conditions"],
                                               {"Flood over 3cm. deep": "Flood_3cm_plus"})


In [8]:
df.head(5)

,Accident Date,Day_of_Week,Junction_Control,Junction_Detail,Accident_Severity,Light_Conditions,Number_of_Casualties,Number_of_Vehicles,Road_Surface_Conditions,Road_Type,Speed_limit,Time,Urban_or_Rural_Area,Weather_Conditions
0,2021-01-01,Friday,Give way or uncontrolled,T or staggered junction,Serious,Daylight,1.0,2.0,Dry,One way street,30.0,15:11,Urban,Fine_no_high
1,2021-01-05,Tuesday,Give way or uncontrolled,Crossroads,Serious,Daylight,11.0,2.0,Wet or damp,Single carriageway,30.0,10:59,Urban,Fine_no_high
2,2021-01-04,Monday,Give way or uncontrolled,T or staggered junction,Slight,Daylight,1.0,2.0,Dry,Single carriageway,30.0,14:19,Urban,Fine_no_high
3,2021-01-05,Tuesday,Auto traffic signal,T or staggered junction,Serious,Daylight,1.0,2.0,Frost or ice,Single carriageway,30.0,08:10,Urban,Other
4,2021-01-06,Wednesday,Auto traffic signal,Crossroads,Serious,Dark_lit,1.0,2.0,Dry,Single carriageway,30.0,17:25,Urban,Fine_no_high


**Feature engineering**

In [9]:
# Time_of_Day from Time
if "Time" in df.columns:
    def time_bucket(t):
        if pd.isna(t): return "Unknown"
        hh = int(t.split(":")[0])
        if 5 <= hh < 11:  return "Morning"
        if 11 <= hh < 16: return "Afternoon"
        if 16 <= hh < 20: return "Evening"
        return "Night"
    df["Time_of_Day"] = df["Time"].apply(time_bucket)

# Month / Season / Day_name from Accident Date
if "Accident Date" in df.columns:
    df["Month"] = df["Accident Date"].dt.month
    def season(m):
        if m in [12,1,2]:  return "Winter"
        if m in [3,4,5]:   return "Spring"
        if m in [6,7,8]:   return "Summer"
        if m in [9,10,11]: return "Autumn"
        return "Unknown"
    df["Season"]   = df["Month"].apply(season)
    df["Day_name"] = df["Accident Date"].dt.day_name()

# High speed flag
if "Speed_limit" in df.columns:
    df["High_Speed"] = (pd.to_numeric(df["Speed_limit"], errors="coerce") >= 60).astype(int)


**Bucketing rare categories**

In [10]:
df.head()

,Accident Date,Day_of_Week,Junction_Control,Junction_Detail,Accident_Severity,Light_Conditions,Number_of_Casualties,Number_of_Vehicles,Road_Surface_Conditions,Road_Type,Speed_limit,Time,Urban_or_Rural_Area,Weather_Conditions,Time_of_Day,Month,Season,Day_name,High_Speed
0,2021-01-01,Friday,Give way or uncontrolled,T or staggered junction,Serious,Daylight,1.0,2.0,Dry,One way street,30.0,15:11,Urban,Fine_no_high,Afternoon,1.0,Winter,Friday,0
1,2021-01-05,Tuesday,Give way or uncontrolled,Crossroads,Serious,Daylight,11.0,2.0,Wet or damp,Single carriageway,30.0,10:59,Urban,Fine_no_high,Morning,1.0,Winter,Tuesday,0
2,2021-01-04,Monday,Give way or uncontrolled,T or staggered junction,Slight,Daylight,1.0,2.0,Dry,Single carriageway,30.0,14:19,Urban,Fine_no_high,Afternoon,1.0,Winter,Monday,0
3,2021-01-05,Tuesday,Auto traffic signal,T or staggered junction,Serious,Daylight,1.0,2.0,Frost or ice,Single carriageway,30.0,08:10,Urban,Other,Morning,1.0,Winter,Tuesday,0
4,2021-01-06,Wednesday,Auto traffic signal,Crossroads,Serious,Dark_lit,1.0,2.0,Dry,Single carriageway,30.0,17:25,Urban,Fine_no_high,Evening,1.0,Winter,Wednesday,0


**Enforce numeric dtypes and quick sanity report**

In [11]:
for c in ["Speed_limit","Number_of_Vehicles","Number_of_Casualties","Month"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n=== Post-cleaning snapshot ===")
cols_to_peek = ["Accident_Severity","Day_of_Week","Light_Conditions","Weather_Conditions",
                "Road_Surface_Conditions","Road_Type","Junction_Control","Junction_Detail",
                "Urban_or_Rural_Area"]
for col in cols_to_peek:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique")
        print(df[col].value_counts(dropna=False).head(5).to_dict(), "\n")



=== Post-cleaning snapshot ===
Accident_Severity: 3 unique
{'Slight': 12264, 'Serious': 1429, 'Fatal': 352} 

Day_of_Week: 7 unique
{nan: 8388, 'Saturday': 950, 'Thursday': 930, 'Friday': 877, 'Wednesday': 852} 

Light_Conditions: 5 unique
{'Daylight': 10026, 'Dark_lit': 3943, 'Dark_unknown': 33, 'Dark_none': 22, 'Dark_unlit': 21} 

Weather_Conditions: 9 unique
{'Fine_no_high': 12019, 'Rain_no_high': 1415, 'Other': 219, 'Snow_no_high': 110, 'Rain_high_winds': 99} 

Road_Surface_Conditions: 6 unique
{'Dry': 11110, 'Wet or damp': 2695, 'Frost or ice': 134, 'Snow': 100, 'Flood_3cm_plus': 5} 

Road_Type: 6 unique
{'Single carriageway': 11691, 'Dual carriageway': 1503, 'Roundabout': 493, 'One way street': 287, 'Slip road': 70} 

Junction_Control: 5 unique
{'Give way or uncontrolled': 7576, 'Unknown': 3308, 'Auto traffic signal': 3133, 'Stop sign': 17, 'Authorised person': 11} 

Junction_Detail: 9 unique
{'T or staggered junction': 6737, 'Not at junction': 3308, 'Crossroads': 2615, 'Roundab

**Definiing input(x) and target(y) features**

In [12]:
assert "Accident_Severity" in df.columns
y = df["Accident_Severity"].astype("category")

candidate_X = [
    "Day_name","Time_of_Day","Month","Season",
    "Weather_Conditions","Light_Conditions","Road_Surface_Conditions",
    "Road_Type","Junction_Detail","Junction_Control",
    "Urban_or_Rural_Area","Speed_limit","High_Speed",
]
X_cols = [c for c in candidate_X if c in df.columns]
X = df[X_cols].copy()


assert "Number_of_Casualties" not in X.columns
assert "Accident_Severity" not in X.columns

print("X (classification):", X_cols)
print("y distribution:\n", y.value_counts(normalize=True).round(3))


X (classification): ['Day_name', 'Time_of_Day', 'Month', 'Season', 'Weather_Conditions', 'Light_Conditions', 'Road_Surface_Conditions', 'Road_Type', 'Junction_Detail', 'Junction_Control', 'Urban_or_Rural_Area', 'Speed_limit', 'High_Speed']
y distribution:
 Accident_Severity
Slight     0.873
Serious    0.102
Fatal      0.025
Name: proportion, dtype: float64


**Split x and y and stratify bt class**

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


**Building preprocessors**

In [14]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X_cols if c not in cat_cols]

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

num_pipe_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

num_pipe_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preproc_linear = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe_linear, num_cols),
    ],
    remainder="drop"
)

preproc_tree = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe_tree, num_cols),
    ],
    remainder="drop"
)

**Fit and transform**

In [15]:
preproc_linear.fit(X_train)
preproc_tree.fit(X_train)

#For linear classification models
Xtr_lin = preproc_linear.transform(X_train)
Xte_lin = preproc_linear.transform(X_test)

#for tree classification models
Xtr_tree = preproc_tree.transform(X_train)
Xte_tree = preproc_tree.transform(X_test)

print("Linear shapes:", Xtr_lin.shape, Xte_lin.shape)
print("Tree shapes:  ", Xtr_tree.shape, Xte_tree.shape)

Linear shapes: (11236, 58) (2809, 58)
Tree shapes:   (11236, 58) (2809, 58)


**Logistic Regression**

In [17]:
# ===============================================
# Logistic Regression for Accident Severity
# - Reads:  "Road Accident Data.csv"
# - Target: "Accident_Severity"
# - Saves:  artifacts/logreg_pipeline.pkl (end-to-end pipeline)
#           artifacts/logreg_metrics.json
#           artifacts/logreg_confusion_matrix.png
# ===============================================

import os, json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    balanced_accuracy_score, f1_score
)
import joblib
import matplotlib.pyplot as plt

# ---------------------------
# Config
# ---------------------------
CSV_PATH     = "Road Accident Data.csv"   # keep this name if your file matches
OUTDIR       = "artifacts"
TEST_SIZE    = 0.20
RANDOM_STATE = 42

os.makedirs(OUTDIR, exist_ok=True)

# ---------------------------
# Load data
# ---------------------------
df = pd.read_csv(CSV_PATH)
TARGET = "Accident_Severity"

# ---------------------------
# Minimal feature engineering to keep it aligned & fast
# ---------------------------
# Extract month from 'Accident Date' (dataset uses day-first format)
df["Month"] = pd.to_datetime(df["Accident Date"], dayfirst=True, errors="coerce").dt.month

# Columns we do not feed as features
drop_cols = ["Accident_Index", "Accident Date", "Time"]

# Numeric & categorical partitions
numeric_cols = [
    "Latitude", "Longitude", "Number_of_Casualties",
    "Number_of_Vehicles", "Speed_limit", "Month"
]
feature_cols     = [c for c in df.columns if c not in drop_cols + [TARGET]]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print("Target:", TARGET)
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# ---------------------------
# Preprocessing pipelines
# ---------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # Use sparse=True for compatibility across scikit-learn versions
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# ---------------------------
# Model: multinomial LR with 'saga' (good for large sparse OHE)
# ---------------------------
logreg = LogisticRegression(
    solver="saga",
    multi_class="multinomial",
    max_iter=1000,
    n_jobs=-1,
    class_weight="balanced",  # important if classes are imbalanced
    penalty="l2",
    C=1.0,
    random_state=RANDOM_STATE
)

pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("clf", logreg)
])

# ---------------------------
# Train / Test Split
# ---------------------------
X = df[feature_cols]
y = df[TARGET].astype(str)  # keep original labels like 'Fatal','Serious','Slight'

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# ---------------------------
# Fit
# ---------------------------
pipe.fit(X_train, y_train)

# ---------------------------
# Evaluate
# ---------------------------
y_pred   = pipe.predict(X_test)
acc      = accuracy_score(y_test, y_pred)
bal_acc  = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
report   = classification_report(y_test, y_pred, digits=4)

print("\n=== Logistic Regression (multinomial, saga) ===")
print(f"Accuracy:           {acc:.4f}")
print(f"Balanced Accuracy:  {bal_acc:.4f}")
print(f"Macro F1:           {macro_f1:.4f}")
print("\nClassification Report:\n", report)

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("Classes:", labels)
print("Confusion Matrix:\n", cm)

# ---------------------------
# Save artifacts
# ---------------------------
model_path = os.path.join(OUTDIR, "logreg_pipeline.pkl")
joblib.dump(pipe, model_path)
print(f"\nSaved model pipeline to: {model_path}")

metrics = {
    "accuracy": acc,
    "balanced_accuracy": bal_acc,
    "macro_f1": macro_f1,
    "classes": labels
}
metrics_path = os.path.join(OUTDIR, "logreg_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to: {metrics_path}")

# Confusion matrix figure
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Logistic Regression – Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
cm_path = os.path.join(OUTDIR, "logreg_confusion_matrix.png")
plt.savefig(cm_path)
plt.close(fig)
print(f"Saved confusion matrix figure to: {cm_path}")

Target: Accident_Severity
Numeric columns: ['Latitude', 'Longitude', 'Number_of_Casualties', 'Number_of_Vehicles', 'Speed_limit', 'Month']
Categorical columns: ['Day_of_Week', 'Junction_Control', 'Junction_Detail', 'Light_Conditions', 'Local_Authority_(District)', 'Carriageway_Hazards', 'Police_Force', 'Road_Surface_Conditions', 'Road_Type', 'Urban_or_Rural_Area', 'Weather_Conditions', 'Vehicle_Type']


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



=== Logistic Regression (multinomial, saga) ===
Accuracy:           0.5300
Balanced Accuracy:  0.5693
Macro F1:           0.2622

Classification Report:
               precision    recall  f1-score   support

       Fatal     0.0368    0.4455    0.0680       211
       Fetal     0.0330    0.9000    0.0636        10
     Serious     0.1669    0.3784    0.2316      1895
      Slight     0.9019    0.5532    0.6858     12958

    accuracy                         0.5300     15074
   macro avg     0.2846    0.5693    0.2622     15074
weighted avg     0.7968    0.5300    0.6196     15074

Classes: ['Fatal', 'Fetal', 'Serious', 'Slight']
Confusion Matrix:
 [[  94    9   65   43]
 [   1    9    0    0]
 [ 409   32  717  737]
 [2051  223 3515 7169]]

Saved model pipeline to: artifacts/logreg_pipeline.pkl
Saved metrics to: artifacts/logreg_metrics.json
Saved confusion matrix figure to: artifacts/logreg_confusion_matrix.png


**Saving Model**

In [19]:
import pandas as pd, joblib

PIPELINE_PATH = "artifacts/logreg_pipeline.pkl"
DATA_PATH     = "Road Accident Data.csv"
TARGET        = "Accident_Severity"

pipe = joblib.load(PIPELINE_PATH)
df  = pd.read_csv(DATA_PATH)

# Build a small sample input (drop non-feature cols & target)
X = df.drop(columns=[c for c in ["Accident_Index", "Accident Date", "Time", TARGET] if c in df.columns], errors="ignore").head(5)

# Recreate 'Month' column for the sample data
X["Month"] = pd.to_datetime(df["Accident Date"].head(5), dayfirst=True, errors="coerce").dt.month


pred  = pipe.predict(X)
proba = pipe.predict_proba(X)

print("Predictions:", pred)
print("Class order:", pipe.classes_)
print("Probabilities (row 0):", proba[0])

Predictions: ['Fetal' 'Fatal' 'Fetal' 'Slight' 'Fetal']
Class order: ['Fatal' 'Fetal' 'Serious' 'Slight']
Probabilities (row 0): [7.16560755e-10 9.99999999e-01 1.59406519e-10 1.53375592e-10]
